
这是一个使用 PyTorch 和 NCCL (NVIDIA Collective Communications Library) 实现的简化版 TP=2 矩阵乘法示例。
- 在实际框架（如 Megatron-LM）中，这些逻辑被封装在底层 C++/CUDA 算子中，但核心逻辑完全如下所示：

1. 初始化: 获取当前 rank (0 或 1)，建立通信组。
2. 权重切分: Rank 0 拿左半边，Rank 1 拿右半边。
3. 前向传播:
    - 本地计算Y_local = X * W_local
    - All-Gather:交换结果，拼接完整的Y
4. 反向传播（pytorch自动处理大部分，但我们需要理解梯度流向）
    - 接收上游梯度dY
    - Reduce-Scatter(隐含在All-Gather的反向中)：将dY切分，各自算权重梯度
    - All-Reduce(隐含)：累加输入梯度dX

至少需要2张GPU才能运行此代码

运行命令： torchrun --nproc_per_node=2 tp_matnul_example.py

- <b>进程组 (Process Group)</b>	所有参与分布式训练的进程的集合。	你使用2台机器，每台机器4个GPU，那么总共就有8个进程，它们共同构成一个全局进程组，也称为 World。
- <b>世界大小 (World Size)</b>	进程组中进程的总数。	接上例，世界大小就是8。
- <b>排名 (Rank)</b>	给进程组中每个进程的唯一标识符，是一个从0到（世界大小-1）的整数。	主进程通常被分配为Rank 0，它在初始化、保存日志和模型时承担特殊的协调职责。
- <b>本地排名 (Local Rank)</b>	一个节点（即一台机器）内部的进程排名。	在第一个节点（Node 0）上，其4个GPU进程的本地排名是0,1,2,3。在第二个节点（Node 1）上，也是0,1,2,3。

In [ ]:
import torch
import torch.distributed as dist
import torch.nn as nn
import os

def setup():
    '''初始化分布式环境'''
    dist.init_process_group('nccl') # 初始化进程组
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    print(f'rank {rank} 初始化完成，总卡数：{world_size}')
    return rank,world_size

def cleanup():
    '''清理分布式环境'''
    dist.destroy_process_group()
    
class ColumnParallelLinear(nn.Moudle):
    '''
    模拟TP=2的列并行线性层column parallel linear
    对应公式：Y = X * W
    策略：W 按列切分
    '''
    
    def __init__(self,in_features,out_features,rank,world_size):
        super().__init__()
        self.rank = rank
        self.world_size = world_size
        
        # 1.验证配置
        assert world_size == 2,'本例仅演示TP=2'
        assert out_features % world_size == 0 ,'输出维度必须能被TP整除'

        self.in_features = in_features
        self.out_features_per_partition = out_features // world_size # 输出：每个GPU负责的输出特征数量
        '''
        #   假设一个线性层：输入1000维，输出2048维
            out_features = 2048
            world_size = 4  # 使用4个GPU

            out_features_per_partition = 2048 // 4 = 512

            # 结果：
            # GPU 0: 负责输出特征 [0:512]
            # GPU 1: 负责输出特征 [512:1024]  
            # GPU 2: 负责输出特征 [1024:1536]
            # GPU 3: 负责输出特征 [1536:2048]
        '''
        
        # 定义本地权重（只包含一部分）
        # 如果out_features = 8,TP=2,则每个卡只有4列
        # rank 0 : [0：4]  rank 1: [4:8]
        start_idx = rank * self.out_features_per_partition
        end_idx = start_idx + self.out_features_per_partition
        
        # 初始化完整权重用于参考，然后切片
        # 实际训练中直接初始刷切片后的权重
        full_weight = torch.randn(in_features,out_features,requires_gard=True)
        '''
        local_weight_data是一个独立的副本，没有梯度历史的张量
        
        每个GPU拥有自己的权重副本！如果不clone，所有GPU会共享同一个full_weight的视图，
        导致一个GPU的更新影响其他GPU
        
        detach() 返回一个新的张量，从当前计算图中分离出来，不再追踪其梯度历史：是一个独立的参数，没有历史依赖
        如果不detach，weight会继承full_weight的梯度历史
        '''
        local_weight_data = full_weight[:,start_idx:end_idx].clone().detach() # 选择所有行，只选择start_idx ~end_idx之间的列
        
        # 注册为参数（这样optimizer只会更新这一部分）
        self.weight = nn.Parameter(local_weight_data)
        
        # 偏置（通常只在最后一个TP层后者特定策略下处理，这里简化为每卡都有局部偏置）
        # 在megatron中，偏置通常只在rank 0 初始化或者最后gather后处理，这里为了演示简单化
        self.bias = None
        
def forward(self,x):
    '''
    x:[batch,in_features]
    注意：在TP中，输入x通常在所有卡上是完整的（副本）
    '''
    # --- 步骤 1：本地矩阵乘法 ---
    # rank 0 : x * W_left  -> Y_left
    # rank 1 : x * W_right -> Y_right
    y_local = torch.matmul(x,self.weight)
    
    # --- 步骤 2：All-Gather(聚合结果) --- 
    # 目标：让每张卡都有完整的Y = [Y_left,Y_right]
    # 常见一个存放完整结果的tensor
    batch_size = x.shape[0]
    y_full = torch.empty(
        batch_size,
        self.out_features_per_partition * self.world_size,
        device = x.device,
        dtype = x.dtype
    )
    '''
    
    
    执行All-Gather
    input_tensors: 当前进程持有的局部小张量
    output_tensors: 存放结果的完整大张量
        如果 input 是 [2, 4]，world_size=2，那么 output 必须是 [2, 8]
    group: 通信组
    
    dist.all_gather_into_tensor(output_tensor, input_tensor, group=None, async_op=False)
    作用：将集群中所有进程（GPU）持有的数据片段，收集并拼接成一个完整的张量，且每个进程最终都拥有这份完整的数据。
    
    在 TP=2 的例子中，它负责把 g0 算出的左半部分结果和 g1 算出的右半部分结果拼起来，让两张卡都拿到完整的输出。
    
    拼接： 
        Rank 0 将收到的 B 拼在自己持有的 A 后面 →→ [A, B] 写入 output_0。
        Rank 1 将收到的 A 拼在自己持有的 B 前面 (或按顺序) →→ [A, B] 写入 output_1。
        注意：拼接顺序严格按照 Rank ID 排序 (Rank 0 的数据在最前，Rank 1 紧随其后)。
        
    调用后 (状态)：
        Rank 0: output_0 包含 [A, B], 形状 [2, 8]
        Rank 1: output_1 包含 [A, B], 形状 [2, 8]
        结果：所有卡现在都有了完全相同的一份完整数据。
    '''
    dist.all_gather_into_tensor(y_full,y_local,group=dist.group.WORLD)
    
    # 此时，rank 0 和 rank 1 的y_full都是一模一样的完整结果
    return y_full

def main():
    rank,world_size = setup()
    device = torch.device(f"cuda:{rank}")
    
    # 设置随机种子以保证初始权重可复现
    torch.manual_seed(42 + rank)
    
    # 参数设置
    BATCH_SIZE = 2
    HIDDEN_DIM = 8 
    TP_SIZE = 2
    
    print(f"rank {rank} 创建输入数据X：[{BATCH_SIZE},{HIDDEN_DIM}]")
    
    # 创建输入x（在所有卡上是一样的，模拟embedding输出或上一层的输出
    x = torch.randn(BATCH_SIZE,HIDDEN_DIM,device=device)
    
    # 为了让结果明显，我们手动设置x的值（可选）
    # x.fill_(1.0)
    
    print(f"[rank {rank}]初始化TP线性层（out={HIDDEN_DIM},TP={TP_SIZE}）")

    layer = ColumnParallelLinear(HIDDEN_DIM,HIDDEN_DIM,rank,TP_SIZE).to(device)
    
    # ---前向传播---
    print(f"[rank {rank}] 开始前向传播...")
    output = layer(x)
    
    # 打印结果验证
    # 只有rank 0 打印，避免日志混乱
    if rank == 0 :
        print("\n--- 前向传播结果 (Rank 0 视角) ---")
        print(f"输出形状: {output.shape}") # 应该是 [2, 8]
        print(f"输出内容:\n{output}")
        print("注意: Rank 1 的输出应该与 Rank 0 完全一致！\n")
    
    # 其他进程等待rank 0 
    # 所有进程都必须调用
    dist.barrier()
    
    if rank == 1:
        print(f"--- 前向传播结果 (Rank 1 视角) ---")
        print(f"输出形状: {output.shape}")
        print(f"输出内容:\n{output}")
        print("验证: 是否与 Rank 0 一致？\n")
        
    # --- 反向传播 ---
    print(f"[Rank {rank}] 开始反向传播...")
    # 创建一个假的 Loss 梯度 (假设 Loss 对输出的梯度)
    # 形状必须和 output 一致
    grad_output = torch.ones_like(output, device=device)
    
    # 执行反向
    # 中间层反向 / 分布式 TP (必须传grad_output) 
    output.backward(grad_output)
    
    # 检查梯度
    if rank == 0:
        print("--- 梯度检查结果 (Rank 0) ---")
        print(f"权重梯度形状: {layer.weight.grad.shape}") # 应该是 [8, 4] (本地部分)
        print(f"权重梯度内容:\n{layer.weight.grad}")
        # 理论上，input 的梯度 dX 在所有卡上应该也是一致的且完整的
        # PyTorch 会自动处理 All-Reduce 来保证 x.grad 是完整的
        print(f"输入梯度 X.grad 形状: {x.grad.shape}") # 应该是 [2, 8]
        
    dist.barrier()
    if rank == 1:
        print("--- 梯度检查结果 (Rank 1) ---")
        print(f"权重梯度形状: {layer.weight.grad.shape}")
        print(f"权重梯度内容:\n{layer.weight.grad}")
        print(f"输入梯度 X.grad 形状: {x.grad.shape}")

    cleanup()

if __name__ == "__main__":
    main()

| 特性 | `loss.backward()` | `output.backward(grad_output)` |
| :--- | :--- | :--- |
| 适用对象 | 标量 (Scalar, 0-dim tensor) | 任意形状张量 (Vector/Matrix) |
| 隐含梯度 | 默认为 `1.0` | 必须显式提供 `grad_output` |
| 典型场景 | 标准训练循环结尾 | 自定义反向逻辑、分布式手动步、中间层调试 |
| TP 中的作用 | 不直接用于 TP 层内部 | 核心驱动：驱动梯度穿过 `all_gather` 和 `matmul` |



## 带超时的barrier

In [3]:
# PyTorch 2.0+ 支持 timeout
from datetime import timedelta

# 设置 30 秒超时，防止永久死锁
dist.barrier(timeout=timedelta(seconds=30))

NameError: name 'dist' is not defined

In [ ]:
 # === 场景：同步训练阶段 ===
    for epoch in range(3):
        dist.barrier()  # 同步 epoch 开始
        
        # 模拟训练
        print(f"Rank {rank}: Epoch {epoch} 训练中...")
        
        dist.barrier()  # 确保所有人完成训练
        
        # 只有 rank 0 打印汇总信息
        if rank == 0:
            print(f"=== Epoch {epoch} 全部完成 ===")

- 第一个 barrier：是发令枪。确保所有卡都准备好了，再一起开始跑（防止有的卡跑得太快，数据还没准备好）。
- 第二个 barrier：是集结哨。确保所有卡都跑完了，再一起休息或进行下一步（防止有的卡还没算完，另一张卡就开始打印结果或读取新数据了）。